# 03 · Generator
Mapping z->w, band-limited Fourier input, alias-free synthesis layers, toRGB.

In [ ]:
import os, sys
# ---- platform auto-detect: the same notebook runs on Colab and Kaggle ----
PLATFORM = "kaggle" if os.path.exists("/kaggle/input") else "colab"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if (PLATFORM == "colab" or PLATFORM == "kaggle") and not os.path.exists("src"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Ravikishore710/styleforge3-T.git"], check=True)
    os.chdir("styleforge3-T")
sys.path.insert(0, os.path.abspath("."))
!pip install -q -r requirements.txt
import tensorflow as tf
print("platform:", PLATFORM, "| TF:", tf.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU"))
# Kaggle: enable GPU (Settings -> Accelerator -> GPU P100) and add the FFHQ
# dataset to /kaggle/input, or run scripts/prepare_ffhq.py --source folder.

In [ ]:
from src.config import load_config
from src.generator.generator import Generator
import tensorflow as tf, matplotlib.pyplot as plt
cfg = load_config('configs/ffhq_64.yaml')
G = Generator(cfg)
z = tf.random.normal([4, cfg['z_dim']])
img, ws = G(z, return_ws=True)
print('images:', img.shape, '| ws:', ws.shape, '| G params:', f'{G.count_params():,}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, im in zip(axes, img.numpy()):
    ax.imshow((im + 1) / 2); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
for layer in G.synthesis.layers:
    fu = 0 if layer.fu is None else len(layer.fu)
    fd = 0 if layer.fd is None else len(layer.fd)
    print(f'{layer.name:>10} {layer.in_res:>4}->{layer.out_res:<4} '
          f'critical={layer.critical} fu_taps={fu} fd_taps={fd}')

In [ ]:
import numpy as np
z = tf.constant(np.random.default_rng(0).standard_normal([1, 512], dtype=np.float32))
base = G(z, noise_mode='const', training=False).numpy()
shifted = G(z, noise_mode='const', shift=(8.0, 0.0), training=False).numpy()
err_grid = np.abs(shifted - np.roll(base, 8, axis=1)).mean()
err_self = np.abs(base - np.roll(base, 8, axis=1)).mean()
print(f'grid-shift error {err_grid:.4f} vs naive-roll error {err_self:.4f}')
assert err_grid < err_self